In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "PAQ_E.xpt"
csv_file = DATA_DIR / "PAQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Preserve untouched source for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 PAQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PAQ560",
    "PAD590",
    "PAD600",
    "PAQ605",
    "PAQ610",
    "PAD615",
    "PAQ620",
    "PAQ625",
    "PAD630",
    "PAQ635",
    "PAQ640",
    "PAD645",
    "PAQ650",
    "PAQ655",
    "PAD660",
    "PAQ665",
    "PAQ670",
    "PAD675",
    "PAD680",
    "PAAQUEX"
]


if df.shape != (9359, 21):
    raise ValueError(
        f"Unexpected PAQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "PAQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: PAQ_E structure = "
    "9,359 rows x 21 columns."
)


# ============================================================
# 4. NUMERIC FIELD CHECK
# ============================================================

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


if numeric_cols != EXPECTED_COLUMNS:
    raise ValueError(
        "Unexpected nonnumeric PAQ_E field found."
    )


print(
    "PASS: All 21 released PAQ_E fields are numeric."
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "PAQ560": 69,
    "PAD590": 239,
    "PAD600": 645,
    "PAD680": 8
}


EXPECTED_TINY_TOTAL = 961


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != EXPECTED_TINY_TOTAL:
    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


if true_zero_total != 0:
    raise ValueError(
        "Unexpected true numeric zeros already "
        "decoded in source XPT."
    )


print(
    "\nPASS: Exact tiny pattern confirmed."
)

print(
    "      PAQ560 = 69"
)

print(
    "      PAD590 = 239"
)

print(
    "      PAD600 = 645"
)

print(
    "      PAD680 = 8"
)

print(
    "      TOTAL  = 961"
)


# ============================================================
# 7. CDC ZERO-SAFETY CHECK
# ============================================================

#
# CDC:
#
# PAQ560
# IF NEVER, ENTER 0
# Range = 0-35
#
# PAD590
# 0 = Less than 1 hour
#
# PAD600
# 0 = Less than 1 hour
#
# PAD680
# Range = 0-1200 minutes
#
# Therefore ONLY these exact tiny values may
# be restored to zero.
#

APPROVED_ZERO_COLUMNS = {
    "PAQ560",
    "PAD590",
    "PAD600",
    "PAD680"
}


if (
    set(actual_tiny_counts)
    !=
    APPROVED_ZERO_COLUMNS
):
    raise ValueError(
        "STOP: Tiny values found outside "
        "approved zero-capable variables."
    )


print(
    "PASS: Tiny values occur only in "
    "CDC zero-permitted variables."
)


# ============================================================
# 8. RESTORE VALIDATED ZEROS
# ============================================================

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, expected_count in (
    EXPECTED_TINY_COUNTS.items()
):

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 961 validated zeros restored."
)


# ============================================================
# 9. VERIFY EXACT ZERO COUNTS
# ============================================================

EXPECTED_ZERO_COUNTS = {
    "PAQ560": 69,
    "PAD590": 239,
    "PAD600": 645,
    "PAD680": 8
}


for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: final zero count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


final_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


if final_zero_total != 961:
    raise ValueError(
        "Unexpected zero(s) introduced "
        "outside approved fields."
    )


print(
    "PASS: Final zero counts exact."
)

print(
    "PASS: No unexpected zeros introduced."
)


# ============================================================
# 10. VERIFY NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:
    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:
    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:
    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):
    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 12. FRACTIONAL / DECIMAL AUDIT
# ============================================================

#
# IMPORTANT:
#
# After the validated zero restorations,
# the attached PAQ_E has NO genuine fractional
# variables.
#
# This is specific to PAQ_E.
#
# Genuine decimal values in dietary/lab/etc.
# files remain AS IS.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:
        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:
    raise ValueError(
        "STOP: Unexpected genuine decimal "
        "values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 13. CDC CHECK - PAQ560
# ============================================================

#
# CDC:
#
# 0-35 = 1,929
# 777  = 0
# 999  = 3
# Missing = 7,427
#
# Actual zero count after restoration = 69
#

s = df["PAQ560"]


if int(
    s.between(
        0,
        35
    ).sum()
) != 1929:
    raise ValueError(
        "PAQ560 0-35 count differs."
    )


if int(
    s.eq(0).sum()
) != 69:
    raise ValueError(
        "PAQ560 zero count differs."
    )


if int(
    s.eq(777).sum()
) != 0:
    raise ValueError(
        "Unexpected PAQ560=777."
    )


if int(
    s.eq(999).sum()
) != 3:
    raise ValueError(
        "PAQ560=999 count differs."
    )


if int(
    s.isna().sum()
) != 7427:
    raise ValueError(
        "PAQ560 missing count differs."
    )


print(
    "\nPASS: PAQ560 matches CDC."
)

print(
    "      0-35 = 1,929"
)

print(
    "      of which zero = 69"
)

print(
    "      999 = 3"
)

print(
    "      Missing = 7,427"
)


# ============================================================
# 14. CDC CHECK - PAD590
# ============================================================

EXPECTED_PAD590 = {
    0: 239,
    1: 462,
    2: 529,
    3: 315,
    4: 163,
    5: 193,
    6: 30,
    99: 1
}


actual = (
    df["PAD590"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PAD590:
    raise ValueError(
        "PAD590 frequency mismatch.\n"
        f"Expected: {EXPECTED_PAD590}\n"
        f"Actual:   {actual}"
    )


if int(
    df["PAD590"]
    .isna()
    .sum()
) != 7427:
    raise ValueError(
        "PAD590 missing count differs."
    )


print(
    "PASS: PAD590 matches CDC."
)


# ============================================================
# 15. CDC CHECK - PAD600
# ============================================================

EXPECTED_PAD600 = {
    0: 645,
    1: 396,
    2: 160,
    3: 71,
    4: 20,
    5: 51,
    6: 585,
    99: 4
}


actual = (
    df["PAD600"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PAD600:
    raise ValueError(
        "PAD600 frequency mismatch.\n"
        f"Expected: {EXPECTED_PAD600}\n"
        f"Actual:   {actual}"
    )


if int(
    df["PAD600"]
    .isna()
    .sum()
) != 7427:
    raise ValueError(
        "PAD600 missing count differs."
    )


print(
    "PASS: PAD600 matches CDC."
)


# ============================================================
# 16. CDC CATEGORICAL CHECKS
# ============================================================

EXPECTED_CATEGORICAL = {

    "PAQ605": (
        {1: 1303, 2: 5802},
        2254
    ),

    "PAQ620": (
        {1: 2646, 2: 4458, 9: 1},
        2254
    ),

    "PAQ635": (
        {1: 1991, 2: 5114},
        2254
    ),

    "PAQ650": (
        {1: 1857, 2: 5247, 9: 1},
        2254
    ),

    "PAQ665": (
        {1: 2661, 2: 4444},
        2254
    ),

    "PAAQUEX": (
        {1: 8749, 2: 610},
        0
    )
}


print(
    "\n--- CDC CATEGORICAL CHECK ---"
)


for col, (
    expected_counts,
    expected_missing
) in EXPECTED_CATEGORICAL.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:
        raise ValueError(
            f"{col} frequency mismatch.\n"
            f"Expected: {expected_counts}\n"
            f"Actual:   {actual_counts}"
        )


    if int(
        df[col]
        .isna()
        .sum()
    ) != expected_missing:
        raise ValueError(
            f"{col} missing count differs."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 17. CDC DAY-COUNT CHECKS
# ============================================================

DAY_CHECKS = {

    "PAQ610": {
        "range_count": 1302,
        "missing": 8056,
        "dk": 1
    },

    "PAQ625": {
        "range_count": 2644,
        "missing": 6713,
        "dk": 2
    },

    "PAQ640": {
        "range_count": 1990,
        "missing": 7368,
        "dk": 1
    },

    "PAQ655": {
        "range_count": 1857,
        "missing": 7502,
        "dk": 0
    },

    "PAQ670": {
        "range_count": 2658,
        "missing": 6698,
        "dk": 3
    }
}


for col, cfg in DAY_CHECKS.items():

    s = df[col]


    if int(
        s.between(
            1,
            7
        ).sum()
    ) != cfg["range_count"]:
        raise ValueError(
            f"{col}: 1-7 count differs."
        )


    if int(
        s.eq(77).sum()
    ) != 0:
        raise ValueError(
            f"Unexpected {col}=77."
        )


    if int(
        s.eq(99).sum()
    ) != cfg["dk"]:
        raise ValueError(
            f"{col}: 99 count differs."
        )


    if int(
        s.isna().sum()
    ) != cfg["missing"]:
        raise ValueError(
            f"{col}: missing count differs."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 18. CDC MINUTE-FIELD CHECKS
# ============================================================

#
# These are questionnaire-entered duration fields,
# not continuous laboratory/dietary measurements.
#
# The released XPT contains whole-minute values only.
#

MINUTE_CHECKS = {

    "PAD615": {
        "low": 10,
        "high": 960,
        "range_count": 1288,
        "dk": 2,
        "missing": 8069
    },

    "PAD630": {
        "low": 10,
        "high": 1440,
        "range_count": 2629,
        "dk": 4,
        "missing": 6726
    },

    "PAD645": {
        "low": 10,
        "high": 600,
        "range_count": 1979,
        "dk": 4,
        "missing": 7376
    },

    "PAD660": {
        "low": 10,
        "high": 990,
        "range_count": 1849,
        "dk": 1,
        "missing": 7509
    },

    "PAD675": {
        "low": 10,
        "high": 720,
        "range_count": 2651,
        "dk": 1,
        "missing": 6707
    }
}


print(
    "\n--- CDC MINUTE FIELD CHECK ---"
)


for col, cfg in MINUTE_CHECKS.items():

    s = df[col]


    if int(
        s.between(
            cfg["low"],
            cfg["high"]
        ).sum()
    ) != cfg["range_count"]:
        raise ValueError(
            f"{col}: valid range count differs."
        )


    if int(
        s.eq(7777).sum()
    ) != 0:
        raise ValueError(
            f"Unexpected {col}=7777."
        )


    if int(
        s.eq(9999).sum()
    ) != cfg["dk"]:
        raise ValueError(
            f"{col}: 9999 count differs."
        )


    if int(
        s.isna().sum()
    ) != cfg["missing"]:
        raise ValueError(
            f"{col}: missing count differs."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 19. CDC CHECK - PAD680
# ============================================================

#
# Minutes sedentary activity
#
# CDC:
#
# 0-1200 = 7,069
# 7777   = 0
# 9999   = 22
# Missing = 2,268
#
# Actual source zero count = 8
#

s = df["PAD680"]


if int(
    s.between(
        0,
        1200
    ).sum()
) != 7069:
    raise ValueError(
        "PAD680 0-1200 count differs."
    )


if int(
    s.eq(0).sum()
) != 8:
    raise ValueError(
        "PAD680 zero count differs."
    )


if int(
    s.eq(7777).sum()
) != 0:
    raise ValueError(
        "Unexpected PAD680=7777."
    )


if int(
    s.eq(9999).sum()
) != 22:
    raise ValueError(
        "PAD680=9999 count differs."
    )


if int(
    s.isna().sum()
) != 2268:
    raise ValueError(
        "PAD680 missing count differs."
    )


print(
    "\nPASS: PAD680 matches CDC."
)

print(
    "      0-1200 = 7,069"
)

print(
    "      of which zero = 8"
)

print(
    "      9999 = 22"
)

print(
    "      Missing = 2,268"
)


# ============================================================
# 20. PUBLIC-RELEASE PROCESSING NOTE
# ============================================================

#
# CDC says respondents with an average of
# 24 hours or more per day of activity had
# these time variables set to missing:
#
# PAD615
# PAD630
# PAD645
# PAD660
# PAD675
# PAD680
#
# The released XPT already contains this processing.
#
# DO NOT reconstruct, fill, or recalculate these values.
#

print(
    "\nPASS: CDC public-release missing-value "
    "processing will remain AS IS."
)


# ============================================================
# 21. BUILD EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL RELEASED XPT
#
# +
# ONLY:
#
# PAQ560 tiny -> 0
# PAD590 tiny -> 0
# PAD600 tiny -> 0
# PAD680 tiny -> 0
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 22. VERIFY ALL FIELDS ARE WHOLE BEFORE INT64
# ============================================================

for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):
        raise ValueError(
            f"STOP: {col} contains a non-whole value."
        )


print(
    "\nPASS: All 21 PAQ_E fields contain "
    "whole-number source values after correction."
)


# ============================================================
# 23. CLEAN INTEGER-STYLE FORMATTING
# ============================================================

#
# Examples:
#
# SEQN
# 41475.0 -> 41475
#
# PAQ560
# tiny -> 0
#
# PAD590
# tiny -> 0
#
# PAD615
# 60.0 -> 60
#
# PAD680
# tiny -> 0
#
# Missing
# NaN -> blank
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All PAQ_E fields converted "
    "to clean integer-style formatting."
)


# ============================================================
# 24. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):
        pre_export_failures.append(
            col
        )


if pre_export_failures:
    raise ValueError(
        "Unexpected pre-export changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY 961 validated zero restorations."
)


# ============================================================
# 25. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
# round()
# float_format
# arbitrary recoding
# recalculation
# missing-value filling
#
# Missing XPT values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 26. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 27. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:
    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 28. EXACT NUMERIC XPT -> CSV FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:
        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:
    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 21 columns numerically match "
    "expected XPT values exactly."
)


# ============================================================
# 29. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:
        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:
    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 30. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 9359:
    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 31. STRICT RAW CSV TOKEN FIDELITY
# ============================================================

#
# PAQ_E has no genuine decimal fields.
#
# Therefore:
#
# nonmissing source value ->
# str(int(expected_source_value))
#
# Missing -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )

    for example in raw_examples:
        print(example)


    raise ValueError(
        "PAQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected source representation."
)


# ============================================================
# 32. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in EXPECTED_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 33. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 34. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:
    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 35. FINAL ZERO RECHECK
# ============================================================

for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"Final {col} zero count changed."
        )


final_zero_total = int(
    csv_df[
        EXPECTED_COLUMNS
    ]
    .eq(0)
    .sum()
    .sum()
)


if final_zero_total != 961:
    raise ValueError(
        "Unexpected final total-zero count."
    )


print(
    "PASS: Final zero counts remain exact."
)


# ============================================================
# 36. FINAL PAD680 RECHECK
# ============================================================

s = csv_df["PAD680"]


if not (
    int(
        s.between(
            0,
            1200
        ).sum()
    ) == 7069

    and

    int(
        s.eq(0).sum()
    ) == 8

    and

    int(
        s.eq(7777).sum()
    ) == 0

    and

    int(
        s.eq(9999).sum()
    ) == 22

    and

    int(
        s.isna().sum()
    ) == 2268
):
    raise ValueError(
        "Final PAD680 distribution changed."
    )


print(
    "PASS: Final PAD680 distribution unchanged."
)


# ============================================================
# 37. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("PAQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "PAQ560 = 69"
)

print(
    "PAD590 = 239"
)

print(
    "PAD600 = 645"
)

print(
    "PAD680 = 8"
)

print(
    "TOTAL  = 961"
)


print(
    "\nGenuine decimal variables:",
    0
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "PAQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 PAQ_E XPT -> CSV
Rows: 9,359
Columns: 21
PASS: PAQ_E structure = 9,359 rows x 21 columns.
PASS: All 21 released PAQ_E fields are numeric.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 961

Tiny values by variable:
PAD600    645
PAD590    239
PAQ560     69
PAD680      8

PASS: Exact tiny pattern confirmed.
      PAQ560 = 69
      PAD590 = 239
      PAD600 = 645
      PAD680 = 8
      TOTAL  = 961
PASS: Tiny values occur only in CDC zero-permitted variables.

--- VALIDATED ZERO RESTORATION ---
PAQ560    : 69 tiny -> 0
PAD590    : 239 tiny -> 0
PAD600    : 645 tiny -> 0
PAD680    : 8 tiny -> 0
PASS: All 961 validated zeros restored.
PASS: Final zero counts exact.
PASS: No unexpected zeros introduced.

Remaining tiny artifacts: 0
PASS: No tiny XPORT artifacts remain.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.